In [ ]:
using KadanoffBaym

import SpecialFunctions: erf, erfc
import LinearAlgebra: diag

using Plots, LaTeXStrings
default(fmt = :png)

## Generating a gaussian pulse

Consider a gaussian pulse
$$\xi(t) = \left(\frac{\Omega^2}{2 \pi}\right)^{1/4}\exp{\left(-\frac{\Omega^2}{4}(t-t_c)^2\right)}$$

Consider a modulator cavity given by the triple $(S_M, L_M, H_M)$

for which $L_M = \lambda(t) a$ where $\lambda(t) \triangleq \frac{\xi(t)}{\sqrt{\int_t^\infty \textrm{d}s |\xi(s)|^2}}$

In [ ]:
# Gaussian pulse parameters
const tc = 15.  # peak arrival time
const Ω = 0.3 # frequency bandwidth of the pulse

function ξ(t)
    return (Ω^2 / (2π))^(1/4) * exp(-(t - tc)^2 * (Ω^2 / 4))
end

# Pump and loss terms
const Γ = 5.0
λ(t::Float64) = iszero(Γ) ? 0.0 : Γ * ξ(t) / sqrt(0.5 * erfc((t - tc) * Ω / sqrt(2)))
γ(t::Float64) = iszero(Γ) ? 0.0 : Γ * ξ(t)

# Photon field Hamiltonian
h(t::Float64) = 5.0 - 0.5im * (λ(t) + γ(t))

# Artificial width of the peak
Σ = 1.0#1e-2

# Initial field occupation
n₀ = 0.0

# Initial conditions
GᵞL = GreenFunction(-1.0im * n₀ * ones(1, 1), Lesser)
GᵞG = GreenFunction(-1.0im * (1 + n₀) * ones(1, 1), Greater)

# Pack them
u0 = [GᵞL, GᵞG];

# Define commutator
⋊(a,b) = a * b - b * adjoint(a)

function f_vert(u, times, t1, t2)
    t = times[t1]
    GL, GG = u[1], u[2]
    
    rhsL = -1.0im * ((h(t) + 1.0im * γ(t) - 1.0im * Σ) * GL[t1,t2])
    rhsG = -1.0im * ((h(t) + 1.0im * γ(t) - 1.0im * Σ) * GG[t1,t2])
    return [rhsL, rhsG]
end

function f_diag(u, times, t1)
    t = times[t1]
    println("t: $t")
    GL, GG = u[1], u[2]
    
    rhsL = -1.0im * (h(t) ⋊ GL[t1,t1] + 1.0im * γ(t) * (GL[t1,t1] + GG[t1,t1]))
    rhsG = -1.0im * (h(t) ⋊ GG[t1,t1] + 1.0im * λ(t) * (GG[t1,t1] - GL[t1,t1]) + 2.0im * γ(t) * GG[t1,t1])
    return [rhsL, rhsG]
end

In [ ]:
tspan = (0.0,30.0)

u, _ = kbsolve(f_vert, f_diag, u0, tspan[1], tspan[2]; atol=1e-7, rtol=1e-5, init_dt=1e-3);
times = u.t;

Plotting

In [ ]:
plot(times, λ.(times), label="Loss term")
plot!(times, γ.(times), label="Pump term")
plot!(twinx(), times, ξ.(times), color=:green, legend=:right, label="Pulse", line=:dot, xlabel=L"t")

In [ ]:
plot(times, GᵞL.data |> diag .|> imag .|> -, label="Lesser")
plot!(times, GᵞG.data |> diag .|> imag .|> -, label="Greater")
plot!(twinx(), times, ξ.(times), color=:green, legend=:right, label="Pulse", line=:dot, xlabel=L"t")

In [ ]:
using Interpolations

itp = interpolate((times,times), GᵞL.data, Gridded(Linear()))
time_eq = range(times[1], stop=times[end], length=length(times))

ρ, (ωs, ts) = wigner_transform([itp(t1,t2) for t1=time_eq, t2=time_eq], ts=time_eq);

heatmap(ts, ωs, real(ρ); xlabel=L"$T$", ylabel=L"$\omega$", ylims=(-10,10))

In [ ]:
f(t,t′) = -1.0im * exp(-1.0im * (t-t′) * (5.0 - 1.0im * sign(t-t′) * Σ)) * ξ(0.5 * (t+t′))

In [ ]:
heatmap([f(a,b) |> real for a in times, b in times])

In [ ]:
heatmap(GᵞL.data |> real)